In [ ]:
from google.colab import drive
drive.mount('/content/drive/',force_remount=True)
%cd /content/drive/MyDrive/Pesquisa/2024_glaucoma/

Mounted at /content/drive/
/content/drive/.shortcut-targets-by-id/1r7Qkk75Cst2DaOKc709zT8_uFCss-MVD/Pesquisa/2024_glaucoma


In [ ]:
import os
import PIL
from PIL import Image, ImageFilter
import random
import shutil
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import csv
import time

In [ ]:
# Verifying NRG and RG quantity on each folder from AIROGS
df = pd.read_csv("./dados/labels.csv")
def contagem_labels_dataset(df):
  acc = 0
  for i in range(0, 90001, 18000):
    print(f"Pasta {acc}")
    if (acc <= 4):
      rg = df.iloc[i:i+18000].loc[df['class'] == 'RG'].shape[0]
      nrg = df.iloc[i:i+18000].loc[df['class'] == 'NRG'].shape[0]
      acc += 1
    else:
      rg = df.iloc[i:].loc[df['class'] == 'RG'].shape[0]
      nrg = df.iloc[i:].loc[df['class'] == 'NRG'].shape[0]
    print(f"RG: {rg}, NRG: {nrg}")

contagem_labels_dataset(df)

Pasta 0
RG: 602, NRG: 17398
Pasta 1
RG: 622, NRG: 17378
Pasta 2
RG: 577, NRG: 17423
Pasta 3
RG: 576, NRG: 17424
Pasta 4
RG: 564, NRG: 17436
Pasta 5
RG: 329, NRG: 11113


# Train, validation and test split

In [ ]:
from torch.utils.data import Dataset
from torchvision.io import decode_image
from PIL import Image
from torchvision.transforms import ToTensor

In [ ]:
## Setting the random seed and enabling determinism
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True, warn_only=True)

In [ ]:
from torch.utils.data import Dataset
from torchvision.io import decode_image
from PIL import Image
from torchvision.transforms import ToTensor

## Custom dataset class creation
# Loads images on the fly, respecting the directory structure
class DatasetGlaucoma(Dataset):
    def __init__(self, img_dir, transform=None, target_transform=None):
        self.img_dir = img_dir
        imgs_total = []

        self.count_nrg = 0
        self.count_rg = 0

        for i in os.listdir(img_dir + "NRG/"):
            imgs_total.append((os.path.join(img_dir + "NRG/", i), torch.tensor(0)))
            self.count_nrg += 1
        for i in os.listdir(img_dir + "RG/"):
            imgs_total.append((os.path.join(img_dir + "RG/", i), torch.tensor(1)))
            self.count_rg += 1

        self.imgs_total = imgs_total
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.imgs_total)

    def len_division(self):
        return f"NRG: {self.count_nrg}, RG: {self.count_rg}"

    def __getitem__(self, idx):
        (image_dir, label) = self.imgs_total[idx]
        with Image.open(image_dir) as img:
            image = img.convert('RGB')
        if self.transform:
                image = self.transform(image)
        return image, label

In [ ]:
import os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import os
import torch
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
import torch.backends.cudnn as cudnn
cudnn.benchmark = True

## Sets the random seed to 123 for reproducibility
SEED = 123
generator = torch.Generator().manual_seed(SEED)
seed_everything(SEED)

## Defines the path to the base dataset
datasetDirectory = "./datasets/discos_15369_original/"

## Instantiates the complete dataset object
dataset_nome = datasetDirectory[datasetDirectory.rindex("/", 0, (len(datasetDirectory) - 1)) + 1:len(datasetDirectory) - 1]
full_dataset = DatasetGlaucoma(
    img_dir=datasetDirectory,
    transform=ToTensor()
)

In [ ]:
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [0.7, 0.2, 0.1], generator=generator)

num_workers = 2
batch_size = 1

train_ds = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=False, num_workers=num_workers)
val_ds = DataLoader(val_dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=num_workers)
test_ds = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=num_workers)
print(len(full_dataset))
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

11732
8213
2346
1173


In [ ]:
from collections import Counter
def organize_classes(ds_set, set_str):
  classes = ["NRG", "RG"]
  counts = Counter()
  for i in ds_set.dataset.indices:
    _, label = full_dataset[i]
    if isinstance(label, torch.Tensor):
      label = label.item()
    counts[label] += 1
    source_path = full_dataset.imgs_total[i][0]
    shutil.copy(source_path, f"./datasets/discos_15369/{set_str}/{classes[label]}/{source_path[source_path.index("recorte"):]}")
  return counts

In [ ]:
test_classes = organize_classes(test_ds, "test")
print(test_classes)

Counter({0: 834, 1: 339})


In [ ]:
val_classes = organize_classes(val_ds, "val")
print(val_classes)

Counter({0: 1710, 1: 636})


In [ ]:
train_classes = organize_classes(train_ds, "train")
print(train_classes)

Counter({0: 5926, 1: 2287})


In [ ]:
def copy_imgs(source, destination):
  for img in os.listdir(source):
    shutil.copy(os.path.join(source, img), os.path.join(destination, img))

In [ ]:
copy_imgs("./datasets/discos_15369_original/train/NRG/", "./datasets/discos_15369/train/NRG/")
copy_imgs("./datasets/discos_15369_original/train/RG/", "./datasets/discos_15369/train/RG/")

copy_imgs("./datasets/discos_15369_original/val/NRG/", "./datasets/discos_15369/val/NRG/")
copy_imgs("./datasets/discos_15369_original/val/RG/", "./datasets/discos_15369/val/RG/")

copy_imgs("./datasets/discos_15369_original/test/NRG/", "./datasets/discos_15369/test/NRG/")
copy_imgs("./datasets/discos_15369_original/test/RG/", "./datasets/discos_15369/test/RG/")

# Segmentation


In [ ]:
!python -m pip install pyyaml==5.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.2/274.2 kB 14.3 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
# !python -m pip install pyyaml==5.1
# !python -m pip install pyyaml
import sys, os, distutils.core
# Note: This is a faster way to install detectron2 in Colab, but it does not include all functionalities.
# See https://detectron2.readthedocs.io/tutorials/install.html for full installation instructions
!git clone 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])}
sys.path.insert(0, os.path.abspath('./detectron2'))

# Properly install detectron2. (Please do not install twice in both ways)
# !python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

fatal: destination path 'detectron2' already exists and is not an empty directory.
Ignoring dataclasses: markers 'python_version < "3.7"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 8.9 MB/s eta 0:00:00
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=d997336b2edf652eb91b37b310492cc937815a34a3ef56a03143e47b0d007585
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
Successfully built fvcore


In [ ]:
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-y1hgwmdv
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-y1hgwmdv
  Resolved https://github.com/facebookresearch/detectron2.git to commit 8a9d885b3d4dcf1bef015f0593b872ed8d32b4ab
  Preparing metadata (setup.py) ... done


In [ ]:
import detectron2
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog
from detectron2 import model_zoo

In [ ]:
# Escolhendo as imagens que ainda não foram segmentadas
classe = "RG" # escolha da classe (a segmentação com NRG e RG foram feitas de forma separada)

imgs_processadas = [nome[nome.index("TRAIN"):-4] for nome in os.listdir(f"./datasets/segmentadas_15369/{classe}")]
imgs_cortar = [nome for nome in os.listdir(f"./datasets/originais_15369/{classe}")]

for img in imgs_processadas:
  try:
    imgs_cortar.remove(img + ".jpg");
  except ValueError:
    continue

In [ ]:
# Selecionando as imagens rotacionadas para segmentar (não foi preciso verificar as já segmentadas, pois foi feito tudo de uma vez)
# Esse caso é para as imagens originais do data augmentation que sofreram rotação em 15, 30, 45 ou 60 graus e precisam da segmentação

# Obs: essa linha só deve ser executada se todas as imagens presentes na pasta devam ser segmentadas. Caso contrário, é melhor selecionar pelo bloco abaixo
imgs_cortar = [nome for nome in os.listdir(f"./datasets/teste_rot")]

In [ ]:
# to_segment representa imagens do data augmentation que foram rotacionadas e precisam de segmentação
# Para achar o to_segment, é necessário ir para o bloco "aug2" da seção de Data Augmentation
imgs_cortar = to_segment

In [ ]:
# Imagens do data augmentation que foram rotacionadas e precisavam de segmentação (faltavam poucas, então defini assim)
# to_segment = ['rot_45_TRAIN054275.jpg',
#               'rot_60_TRAIN011486.jpg',
#               'rot_60_TRAIN022923.jpg',
#               'rot_15_TRAIN071025.jpg',
#               'rot_45_TRAIN030831.jpg']

In [ ]:
# carregando pesos já treinados
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.WEIGHTS = "./TreinamentoSegmentacao/Segmenta_model_final.pth"  # caminho p/ modelo
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7 # para o modelo só mostrar detecções com confiança acima de 70%
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1 #o modelo  só precisa detectar e segmentar uma classe, nesse caso o disco óptico

cfg.MODEL.DEVICE = "cpu"

predictor = DefaultPredictor(cfg) # carrega o modelo

#indicando caminho das imagens
# caminho pode mudar (considerando o data augmentation)
img_folder = f"./datasets/originais_15369/{classe}/"
classe = "RG"
img_paths = [os.path.join(img_folder, f) for f in imgs_cortar]

# Criando um arquivo para armazenar as coordenadas da região proposta
fieldnames = ['nome', 'instance', 'xmin', 'xmax', 'ymin', 'ymax', 'dx', 'dy', 'altura', 'largura']
#with open('coordenadas2.csv', 'w', newline='') as csvfile:
 #   writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
  #  writer.writeheader()

QuantImgs = len(img_paths)
execution_time = 0.0
# loop para segmentar os discos
for img_path in img_paths:
    os.system("clear")
    print(f'imgs: {QuantImgs}, tempo_por_imagem: {execution_time:.2f} s, eta: {execution_time * QuantImgs:.2f} ')
    start_time = time.perf_counter()
    img = cv2.imread(img_path)
    outputs = predictor(img)

    # preprando informações das caixas delimitadoraas
    v = Visualizer(img[:, :, ::-1], MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), scale=1.2)
    instances = outputs["instances"].to("cpu")
    boxes = instances.pred_boxes
    scores = instances.scores
    class_ids = instances.pred_classes

    num_instances = len(boxes)

    dimensoes = img.shape
    altura = dimensoes[0]
    largura = dimensoes[1]

    #indicação da pasta onde ficará os segmentados
    output_dir = f"./datasets/segmentadas_15369/{classe}/"
    # output_dir = f"./datasets/teste_aug_2_2"

    for i in range(num_instances):
        bbox = boxes.tensor[i].numpy()
        score = scores[i].item()
        class_id = class_ids[i].item()

        # desenha caixas
        v.draw_box(bbox, edge_color='red', alpha=score)

        # instruções oara recortar o disco
        x1, y1, x2, y2 = map(int, bbox)
        cropped_img = img[y1:y2, x1:x2]
        cropped_img = cv2.resize(cropped_img, (390, 390))
        # coordenadas = {'nome': os.path.basename(img_path)[:-4], "instance": i, "xmin": x1, "xmax": x2, "ymin": y1, "ymax": y2, "dx": x2-x1, "dy": y2-y1, "altura": altura, "largura": largura}
        # with open('coordenadasFinal.csv', 'a', newline='') as csvfile:
        #   writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        #   writer.writerow(coordenadas)

        # salva imagem recortada
        output_crop_path = os.path.join(output_dir, f"recorte_{i}_" + os.path.basename(img_path)[:-4] + ".png")
        cv2.imwrite(output_crop_path, cropped_img)

    # desenhando as  caixas delimitadoraas
    # out_img = v.get_output().get_image()
    # out_img = cv2.cvtColor(out_img, cv2.COLOR_BGR2RGB)
    # output_annotated_path = os.path.join(output_dir, "marcada_" + os.path.basename(img_path))
    # cv2.imwrite(output_annotated_path, out_img)
    end_time = time.perf_counter()
    execution_time = end_time - start_time
    QuantImgs -= 1

print("Todas as imagens foram processadas com sucesso!")


# Verificação dos recortes segmentados

In [ ]:
# pasta = "./datasets/segmentadas_15369/NRG/"
pasta = "./datasets/teste_aug_2_2"
imgs_pasta = os.listdir(pasta)
recortes_ins1 = [img for img in imgs_pasta if img[img.index("_") + 1] != "0"]
def visualizar_recortes(pasta, tipo_ins):
  fig, axes = plt.subplots(4, 3, figsize=(10, 8))
  axes = axes.flatten()

  for i, imagem in enumerate(recortes_ins1):
    if i < len(axes):
      if (tipo_ins == 0):
        img_path = os.path.join(pasta, "recorte_0_" + imagem[imagem.index("TRAIN"):])
      else:
        img_path = os.path.join(pasta, imagem)
      img = cv2.imread(img_path)
      img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
      axes[i].imshow(img)
      axes[i].set_title(img_path[img_path.index("recorte_"):])

  fig.delaxes(axes[5])

  plt.tight_layout()
  plt.show()

In [ ]:
visualizar_recortes(pasta, 1)

In [ ]:
# Visualizando recortes gerados a partir da mesma imagem mas com instância 0
visualizar_recortes(pasta, 0)

# Data Augmentation

In [ ]:
# TRAIN
#   NRG: 5926
#   RG: 2287 → 5926 (gerar 3639)
#   aug1: 763 -> 1213 - rot[90, 180, 270] + (hue or sat) + flip
#   aug2: 762 -> 1213 - rot[15, 30, 45, 60] + flip + blur
#   aug3: 762 -> 1213 - flip hor + hue + sat + flip vert
# VAL
# 	NRG: 1710
# 	RG: 636
# TEST
# 	NRG: 834
# 	RG: 339

# teste_aug1_2 -> segundo aumento do aug1
# teste_aug2_2 -> segundo aumento do aug2
# teste_aug3_2 -> segundo aumento do aug3

In [ ]:
import json
import torch
import torchvision
import numpy as np

In [ ]:
# original folder
path_rg = "./datasets/discos_15369_original/train/RG"
original_imgs = os.listdir(path_rg)
aug1 = original_imgs[:763]
aug2 = original_imgs[763:1525]
aug3 = original_imgs[1525:]

In [ ]:
# Armazenando as imagens que estão em cada "parte" do data augmentation
aug = {"aug1": aug1, "aug2": aug2, "aug3": aug3}
with open('./aug.json', 'w') as json_file:
  json.dump(aug, json_file)

## Aug1
763 -> 1213 - rot[90, 180, 270] + (hue or sat) + flip

In [ ]:
with open('./aug.json', 'r') as json_file:
  aug = json.load(json_file)

aug1 = aug["aug1"]

In [ ]:
# Aplicando o data augmentation nas 763 imagens originais

rangeRot = [90, 180, 270]
flip = ["hori", "vert"]
process = ["hue", "sat"]
hues = [-0.04, -0.032, -0.024, -0.016, 0.016, 0.024, 0.032, 0.04]
sats = [0.8, 0.9, 1.1, 1.2]

aug1_info = dict()
contador = len(aug1)
for img in aug1:
  print(f'Faltam {contador}')
  image = Image.open(os.path.join(path_rg, img))

  num_rot = int(random.random() * len(rangeRot))
  flip_type = int(random.random() * len(flip))
  proc = int(random.random() * len(process))
  factor = 0

  # rotation
  image = image.rotate(rangeRot[num_rot])

  # horizontal or vertical flip
  if flip_type == 0:
    image = image.transpose(method=Image.FLIP_LEFT_RIGHT)
  else:
    image = image.transpose(method=Image.FLIP_TOP_BOTTOM)

  # hue or saturation modification
  if proc == 0:
    factor = hues[int(random.random() * len(hues))]
    image = torchvision.transforms.functional.adjust_hue(image, factor)
  else:
    factor = sats[int(random.random() * len(sats))]
    image = torchvision.transforms.functional.adjust_saturation(image, factor)

  image.save(os.path.join(path_rg, f"rot_{rangeRot[num_rot]}_{flip[flip_type]}_flip_{process[proc]}_{factor}_" + img))
  aug1_info[img] = {"rot": rangeRot[num_rot],
                    "flip": flip[flip_type],
                    f"{process[proc]}": factor}


  contador = contador - 1

with open('./aug1_info.json', 'w') as json_file:
  json.dump(aug1_info, json_file)


In [ ]:
# Aplicando data augmentation pela segunda vez sobre as imagens originais
rangeRot = [90, 180, 270]
flip = ["hori", "vert"]
flip_index = {"hori": 0, "vert": 1}
process = ["hue", "sat"]
hues = [-0.04, -0.032, -0.024, -0.016, 0.016, 0.024, 0.032, 0.04]
sats = [0.8, 0.9, 1.1, 1.2]

with open('./aug1_info.json', 'r') as json_file:
  old_aug1 = json.load(json_file)

new_aug1_info = dict()
contador = len(aug1[:450])
for img in aug1[:450]:
  print(f'Faltam {contador}')
  original_name = img[img.index("recorte"):]
  image = Image.open(os.path.join(orig_folder, original_name))

  rot = old_aug1[original_name]["rot"]
  num_rot = int(random.random() * len(rangeRot))
  while (num_rot == rot):
    num_rot = int(random.random() * len(rangeRot))
  image = image.rotate(rangeRot[num_rot])

  flip_type = old_aug1[original_name]["flip"]
  flip_i = flip_index[flip_type]
  flip_i = 1 - flip_i
  if flip_i == 0:
    image = image.transpose(method=Image.FLIP_LEFT_RIGHT)
  else:
    image = image.transpose(method=Image.FLIP_TOP_BOTTOM)


  new_factor = 0
  proc = 0
  if "hue" in old_aug1[original_name]:
    proc = 0
    new_factor = hues[int(random.random() * len(hues))]
    factor = old_aug1[original_name]["hue"]
    while (new_factor == factor):
      new_factor = hues[int(random.random() * len(hues))]
    image = torchvision.transforms.functional.adjust_hue(image, new_factor)
  else:
    proc = 1
    new_factor = sats[int(random.random() * len(sats))]
    factor = old_aug1[original_name]["sat"]
    while (new_factor == factor):
      new_factor = sats[int(random.random() * len(sats))]
    image = torchvision.transforms.functional.adjust_saturation(image, new_factor)

  new_aug1_info[original_name] = {
      "rot": rangeRot[num_rot],
      "flip": flip[flip_i],
      f"{process[proc]}": new_factor
  }

  image.save(os.path.join("./datasets/teste_aug1_2", f"rot_{rangeRot[num_rot]}_{flip[flip_i]}_flip_{process[proc]}_{new_factor}_{original_name}"))
  contador -= 1

with open('./aug1.2_info.json', 'w') as json_file:
  json.dump(new_aug1_info, json_file)

## Aug2

Para fazer o data augmentation da 2° parte (rot[15, 30, 45, 60] + flip + blur), é necessário rodar a parte do código que seleciona as imagens, rotaciona, voltar para a etapa de segmentação e depois finalizar aqui a etapa do data augmentation

In [ ]:
with open('./aug.json', 'r') as json_file:
  aug = json.load(json_file)

aug2 = aug["aug2"]

In [ ]:
# Selecionando imagens com rotação de 15, 30, 45 ou 60 que já foram segmentadas e imagens que ainda precisam ser rotacionadas (e posteriormente segmentadas)
path_segmented_rot = "./datasets/segmentadas_rot/"
segmented_rot = os.listdir("./datasets/segmentadas_rot/")
rangerot = [15, 30, 45, 60]

acc = 0
to_process = []
processed = dict()
for img in aug2:
  flag = 0
  original_name = img[img.index("TRAIN"):]
  processed[original_name] = []
  for angle in rangerot:
    if (os.path.isfile(f"{path_segmented_rot}recorte_0_rot_{angle}_{original_name}")):
      flag = 1
      acc += 1
      processed[original_name].append(angle)
  if (not flag):
    processed.pop(original_name)
    to_process.append(original_name)

print(acc)
with open('./processed.txt', 'w') as json_file:
  json.dump(processed, json_file)

with open('./to_process.txt', 'w') as file:
  for name in to_process:
    file.write(f"{name}\n")

757


In [ ]:
to_process = []
with open('to_process.txt', 'r') as file:
  for line in file:
    to_process.append(line[:-1])

In [ ]:
# Rotacionando imagens originais para serem segmentadas posteriormente
df = pd.read_csv("./coordenadasFinal.csv")
path = [os.path.join("./datasets/originais_15369/RG/", f[:-4] + ".jpg") for f in to_process]
contador = len(to_process)
rangeRot = [15, 30, 45, 60]

for img in path:
  try:
    print(f"Faltam {contador} imagens")
    num_rot = int(random.random() * len(rangeRot))
    nome = img[img.index("TRAIN"):-4]
    img_original = cv2.imread(img)
    height = df.loc[df['nome'] == nome, 'altura'].item()
    width = df.loc[df['nome'] == nome, 'largura'].item()
    centerX, centerY = (width//2, height//2)
    matriz_rot = cv2.getRotationMatrix2D((centerX, centerY), rangeRot[num_rot], 1.0)
    rotacionada = cv2.warpAffine(img_original, matriz_rot, (int(width), int(height)))
    cv2.imwrite(f"./datasets/teste_rot/rot_{rangeRot[num_rot]}_{nome}.jpg", rotacionada)
    contador -= 1
  except:
    print("erro", img)

In [ ]:
# Verificando imagens já rotacionadas mas que ainda não foram segmentadas
to_segment = []
for img in os.listdir("./datasets/teste_rot"):
  for name in to_process:
    if name[:-4] in img:
      to_segment.append(img)

In [ ]:
# Aplicação do flip e blur sobre as imagens já rotacionadas e segmentadas
path_segmented_rot = "./datasets/segmentadas_rot/"
segmented_rot = os.listdir("./datasets/segmentadas_rot/")
with open('./processed.txt', 'r') as json_file:
  processed = json.load(json_file)
flip = ["hori", "vert"]
rangeRot = [15, 30, 45, 60]
contador = len(processed)

for img in processed:
  print(f"Faltam {contador} imagens")
  original_name = img[img.index("TRAIN"):-4]
  rot = processed[img][0]

  image = Image.open(f"{path_segmented_rot}recorte_0_rot_{rot}_{original_name}.png")
  flip_type = int(random.random() * len(flip))
  if flip_type == 0:
    image = image.transpose(method=Image.FLIP_LEFT_RIGHT)
  else:
    image = image.transpose(method=Image.FLIP_TOP_BOTTOM)

  image = image.filter(ImageFilter.GaussianBlur(radius=2))
  image.save(os.path.join("./datasets/teste_aug2", f"rot_{rot}_{flip[flip_type]}_flip_blur2_recorte_0_" + img[img.index("TRAIN"):]))
  contador -= 1


Faltam 757 imagens
Faltam 756 imagens
Faltam 755 imagens
Faltam 754 imagens
Faltam 753 imagens
Faltam 752 imagens
Faltam 751 imagens
Faltam 750 imagens
Faltam 749 imagens
Faltam 748 imagens
Faltam 747 imagens
Faltam 746 imagens
Faltam 745 imagens
Faltam 744 imagens
Faltam 743 imagens
Faltam 742 imagens
Faltam 741 imagens
Faltam 740 imagens
Faltam 739 imagens
Faltam 738 imagens
Faltam 737 imagens
Faltam 736 imagens
Faltam 735 imagens
Faltam 734 imagens
Faltam 733 imagens
Faltam 732 imagens
Faltam 731 imagens
Faltam 730 imagens
Faltam 729 imagens
Faltam 728 imagens
Faltam 727 imagens
Faltam 726 imagens
Faltam 725 imagens
Faltam 724 imagens
Faltam 723 imagens
Faltam 722 imagens
Faltam 721 imagens
Faltam 720 imagens
Faltam 719 imagens
Faltam 718 imagens
Faltam 717 imagens
Faltam 716 imagens
Faltam 715 imagens
Faltam 714 imagens
Faltam 713 imagens
Faltam 712 imagens
Faltam 711 imagens
Faltam 710 imagens
Faltam 709 imagens
Faltam 708 imagens
Faltam 707 imagens
Faltam 706 imagens
Faltam 705 i

In [ ]:
# Aplicação do data augmentation pela segunda vez (rotacionando as imagens com ângulo diferente da primeira)
path_source = "./datasets/originais_15369/RG/"
path_dest = "./datasets/teste_rot" # imagens rotacionadas (e que ainda precisam de segmentação)
df = pd.read_csv("./coordenadasFinal.csv")
aug2 = aug["aug2"]
contador = len(aug2[:460])
aug2_2_info = dict()

rangeRot = [15, 30, 45, 60]

for img in aug2[:460]:
  print(f"Faltam {contador} imagens")
  original_name = img[img.index("TRAIN"):-4]
  try:
    new_rot = rangeRot[int(random.random() * len(rangeRot))]
    rot = processed[original_name + '.png'][0]
    while (new_rot == rot):
      new_rot = rangeRot[int(random.random() * len(rangeRot))]

    img_original = cv2.imread(os.path.join("./datasets/originais_15369/RG/", original_name+'.jpg'))
    height = df.loc[df['nome'] == original_name, 'altura'].item()
    width = df.loc[df['nome'] == original_name, 'largura'].item()
    centerX, centerY = (width//2, height//2)
    matriz_rot = cv2.getRotationMatrix2D((centerX, centerY), new_rot, 1.0)

    rotacionada = cv2.warpAffine(img_original, matriz_rot, (int(width), int(height)))
    cv2.imwrite(f"./datasets/teste_rot/rot_{new_rot}_{original_name}.jpg", rotacionada)
    aug2_2_info[original_name] = [new_rot]
    contador -= 1
  except:
    print("erro", img)



with open('./aug2_2_info.json', 'w') as json_file:
  json.dump(aug2_2_info, json_file)


Faltam 460 imagens
Faltam 459 imagens
Faltam 458 imagens
Faltam 457 imagens
Faltam 456 imagens
Faltam 455 imagens
Faltam 454 imagens
Faltam 453 imagens
Faltam 452 imagens
Faltam 451 imagens
Faltam 450 imagens
Faltam 449 imagens
Faltam 448 imagens
Faltam 447 imagens
Faltam 446 imagens
Faltam 445 imagens
Faltam 444 imagens
Faltam 443 imagens
Faltam 442 imagens
Faltam 441 imagens
Faltam 440 imagens
Faltam 439 imagens
Faltam 438 imagens
Faltam 437 imagens
Faltam 436 imagens
Faltam 435 imagens
Faltam 434 imagens
Faltam 433 imagens
Faltam 432 imagens
Faltam 431 imagens
Faltam 430 imagens
Faltam 429 imagens
Faltam 428 imagens
Faltam 427 imagens
Faltam 426 imagens
Faltam 425 imagens
Faltam 424 imagens
Faltam 423 imagens
Faltam 422 imagens
Faltam 421 imagens
Faltam 420 imagens
Faltam 419 imagens
Faltam 418 imagens
Faltam 417 imagens
Faltam 416 imagens
Faltam 415 imagens
Faltam 414 imagens
Faltam 413 imagens
Faltam 412 imagens
Faltam 411 imagens
Faltam 410 imagens
Faltam 409 imagens
Faltam 408 i

In [ ]:
with open('./aug2_2_info.json', 'r') as json_file:
  aug2_2 = json.load(json_file)

img_aug2_2 = os.listdir("./datasets/teste_aug_2_2") # imagens já segmentadas
img_aug2_2 = [img for img in img_aug2_2 if "recorte_0_" in img]

In [ ]:
# Aplicando flip e blur nas imagens rotacionadas e segmentadas pela segunda vez
path_src = "./datasets/teste_aug_2_2" # recortes das imagens que foram rotacionadas pela segunda vez
path_dest = "./datasets/teste_aug2_2"
flip = ["hori", "vert"]
aug222_info = dict()
contador = len(img_aug2_2)
for img in img_aug2_2:
  try:
    image = Image.open(os.path.join(path_src, img))
    rot = img[img.index("rot") + 4: img.index("rot") + 6]
    flip_type = int(random.random() * len(flip))

    if flip_type == 0:
      image = image.transpose(method=Image.FLIP_LEFT_RIGHT)
    else:
      image = image.transpose(method=Image.FLIP_TOP_BOTTOM)

    image = image.filter(ImageFilter.GaussianBlur(radius=2))
    image.save(os.path.join(path_dest, f"rot_{rot}_{flip[flip_type]}_flip_blur2_recorte_0_" + img[img.index("TRAIN"):]))
    aug222_info[img] = {"rot": rot,
                      "flip": flip[flip_type]}
    contador = contador - 1

  except:
    print("erro", img)

with open('./aug222_info.json', 'w') as json_file:
  json.dump(aug222_info, json_file)

## Aug3
762 -> 1213 - horizontal flip + hue modification + saturation modification + vertical flip

In [ ]:
with open('./aug.json', 'r') as json_file:
  aug = json.load(json_file)

aug3 = aug["aug3"]

In [ ]:
# Aplicando data augmentation nas 762 imagens originais
path_source = "./datasets/discos_15369_original/train/RG"
path_dest = "./datasets/discos_15369/train/RG"
hues = [-0.04, -0.032, -0.024, -0.016, 0.016, 0.024, 0.032, 0.04]
sats = [0.8, 0.9, 1.1, 1.2]
aug3_info = dict()
contador = len(aug3)

for img in aug3:
  print(f"Faltam {contador} imagens")
  image = Image.open(os.path.join(path_source, img))
  image = image.transpose(method=Image.FLIP_LEFT_RIGHT)

  hue = hues[int(random.random() * len(hues))]
  sat = sats[int(random.random() * len(sats))]
  image = torchvision.transforms.functional.adjust_hue(image, hue)
  image = torchvision.transforms.functional.adjust_saturation(image, sat)

  image = image.transpose(method=Image.FLIP_TOP_BOTTOM)
  image.save(os.path.join(path_dest, f"hori_flip_hue_{hue}_sat{sat}_vert_flip_recorte_0_" + img[img.index("TRAIN"):]))
  aug3_info[img] = {"hue": hue, "sat": sat}
  contador -= 1

with open('./aug3_info.json', 'w') as json_file:
  json.dump(aug3_info, json_file)

In [ ]:
# Aplicando data augmentation pela segunda vez sobre as imagens originais
with open('./aug3_info.json', 'r') as json_file:
  old_aug3_info = json.load(json_file)

path_source = "./datasets/discos_15369_original/train/RG"
path_dest = "./datasets/discos_15369/train/RG"
hues = [-0.04, -0.032, -0.024, -0.016, 0.016, 0.024, 0.032, 0.04]
sats = [0.8, 0.9, 1.1, 1.2]
aug3_info = dict()
contador = len(aug3[:451])

for img in aug3[:451]:
  print(f"Faltam {contador} imagens")
  original_name = img[img.index("recorte"):]
  image = Image.open(os.path.join(path_source, original_name))
  image = image.transpose(method=Image.FLIP_LEFT_RIGHT)


  new_hue = hues[int(random.random() * len(hues))]
  hue = old_aug3_info[original_name]["hue"]
  while (new_hue == hue):
    new_hue = hues[int(random.random() * len(hues))]
  image = torchvision.transforms.functional.adjust_hue(image, new_hue)

  new_sat = sats[int(random.random() * len(sats))]
  sat = old_aug3_info[original_name]["sat"]
  while (new_sat == sat):
    new_sat = sats[int(random.random() * len(sats))]
  image = torchvision.transforms.functional.adjust_saturation(image, new_sat)


  image = image.transpose(method=Image.FLIP_TOP_BOTTOM)


  image = image.transpose(method=Image.FLIP_LEFT_RIGHT)
  image.save(os.path.join("./datasets/teste_aug3_2", f"hori_flip_hue_{hue}_sat{sat}_vert_flip_hori_flip{original_name}"))
  aug3_info[img] = {"hue": hue, "sat": sat}
  contador -= 1

with open('./aug3_2_info.json', 'w') as json_file:
  json.dump(aug3_info, json_file)